# 02. Линейная алгебра

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/02_linear_algebra.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m02). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Скалярное произведение — главная операция ML

In [ ]:
import numpy as np
a, b = np.array([1., 2, 3]), np.array([2., 0, 1])
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))   # 0.598
# Поиск ближайших эмбеддингов: нормируем один раз, дальше — одно матричное умножение
E = np.random.randn(10_000, 768); E /= np.linalg.norm(E, axis=1, keepdims=True)
q = E[42] + 0.1 * np.random.randn(768); q /= np.linalg.norm(q)
top5 = np.argsort(-(E @ q))[:5]    # 42 будет первым

### SVD — самое полезное разложение

In [ ]:
U0, _ = np.linalg.qr(np.random.randn(512, 256)); V0, _ = np.linalg.qr(np.random.randn(256, 256))
A = U0 @ np.diag(np.exp(-np.arange(256) / 5)) @ V0.T     # матрица с быстро убывающим спектром, как у реальных данных
U, S, Vt = np.linalg.svd(A, full_matrices=False)
k = 20
A_k = U[:, :k] * S[:k] @ Vt[:k]
print(np.linalg.norm(A - A_k) / np.linalg.norm(A))   # ≈ 0.018: ошибка ~2% при 20 компонентах из 256
print((512 * 256), k * (512 + 256 + 1))             # 131072 → 15380 чисел

### PCA — метод главных компонент

In [ ]:
from sklearn.decomposition import PCA
X = np.random.randn(1000, 300)
Z = PCA(n_components=50).fit_transform(X)       # (1000, 50) — то же самое, что SVD выше

### Тензоры и broadcasting

In [ ]:
X = np.random.randn(32, 10)     # батч
mu = X.mean(axis=0)              # (10,)
Xc = X - mu                      # (32,10) - (10,) → работает
# Попарные расстояния без циклов: (N,1,D) - (1,M,D) → (N,M,D)
A, B = np.random.randn(5, 3), np.random.randn(4, 3)
D = np.sqrt(((A[:, None, :] - B[None, :, :])**2).sum(-1))

### Тензоры и broadcasting

In [ ]:
A, B = np.random.randn(3, 4), np.random.randn(4, 5)
Q = K = np.random.randn(2, 8, 16, 64)   # (batch, heads, T, d)
np.einsum('ij,jk->ik', A, B)            # матричное умножение
np.einsum('bhtd,bhsd->bhts', Q, K)      # attention scores по батчу и головам
np.einsum('ii->', A @ A.T)              # след

### 🏋️ Практика модуля

**2.3.** Реализуйте PCA через SVD и сравните со sklearn.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
X = np.random.randn(500, 10) @ np.random.randn(10, 10)
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
Z = Xc @ Vt[:3].T
ratio = S**2 / (S**2).sum()
sk = PCA(3).fit(X)
print(np.allclose(np.abs(Z), np.abs(sk.transform(X))))      # True (знак компоненты произволен!)
print(np.allclose(ratio[:3], sk.explained_variance_ratio_))  # True

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def dot_product():
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    b = np.array([3.0, 0.6])
    for ax, (a, title) in zip(axes, [
        (np.array([2.0, 2.2]), "a·b > 0  (угол < 90°)"),
        (np.array([-0.4, 2.6]), "a·b ≈ 0  (ортогональны)"),
        (np.array([-2.2, 1.2]), "a·b < 0  (угол > 90°)")]):
        arrow(ax, b, C[0], "b")
        arrow(ax, a, C[1], "a")
        proj = (a @ b) / (b @ b) * b
        ax.plot([a[0], proj[0]], [a[1], proj[1]], "--", color="gray")
        arrow(ax, proj, C[2], None)
        cos = a @ b / np.linalg.norm(a) / np.linalg.norm(b)
        ax.set_title(f"{title}\ncos θ = {cos:.2f}")
        ax.set_xlim(-3, 4); ax.set_ylim(-1, 3.2); ax.set_aspect("equal")
    axes[0].text(0.3, -0.7, "зелёная — проекция a на b", color=C[2])
    save(fig, "dot_product")

dot_product()

In [ ]:
def matrix_transform():
    A = np.array([[1.5, 0.8], [0.3, 1.1]])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
    t = np.linspace(0, 2 * np.pi, 200)
    circle = np.stack([np.cos(t), np.sin(t)])
    grid = np.arange(-3, 4)
    for ax, M, title in [(axes[0], np.eye(2), "До: единичная сетка"),
                         (axes[1], A, "После: x ↦ Ax")]:
        for g in grid:
            line = M @ np.stack([np.full(2, g), [-3, 3]])
            ax.plot(*line, color="lightgray", lw=0.8)
            line = M @ np.stack([[-3, 3], np.full(2, g)])
            ax.plot(*line, color="lightgray", lw=0.8)
        ax.fill(*(M @ circle), color=C[0], alpha=0.15)
        ax.plot(*(M @ circle), color=C[0])
        arrow(ax, M @ [1, 0], C[1], "e₁" if M is not A else "Ae₁")
        arrow(ax, M @ [0, 1], C[2], "e₂" if M is not A else "Ae₂")
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
        ax.set_title(title)
    axes[1].text(-2.9, -2.8, f"A = [[1.5, 0.8], [0.3, 1.1]],  det A = {np.linalg.det(A):.2f}\n"
                 "столбцы A — куда переходят e₁ и e₂", fontsize=9)
    save(fig, "matrix_transform")

matrix_transform()

In [ ]:
def eigen():
    A = np.array([[2.0, 1.0], [1.0, 2.0]])
    w, V = np.linalg.eigh(A)
    fig, ax = plt.subplots(figsize=(6, 6))
    rng = np.random.default_rng(0)
    for ang in np.linspace(0, np.pi, 13, endpoint=False):
        v = np.array([np.cos(ang), np.sin(ang)])
        Av = A @ v
        ax.annotate("", xy=Av, xytext=v, arrowprops=dict(arrowstyle="->", color="lightgray"))
        ax.plot(*v, "o", color="gray", ms=3)
    for i, c in zip(range(2), [C[1], C[2]]):
        arrow(ax, V[:, i], c, None)
        arrow(ax, A @ V[:, i], c, f"λ={w[i]:.0f}", alpha=0.6)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
    ax.set_title("Собственные векторы не меняют направление:\nAv = λv  (серые точки — повёрнуты, цветные — только растянуты)")
    save(fig, "eigenvectors")

eigen()

In [ ]:
def pca():
    rng = np.random.default_rng(1)
    X = rng.multivariate_normal([0, 0], [[3, 2.2], [2.2, 2.2]], 400)
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S**2 / (len(X) - 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    ax = axes[0]
    ax.scatter(*Xc.T, s=8, alpha=0.4, color=C[0])
    for i, c in enumerate([C[1], C[2]]):
        arrow(ax, Vt[i] * 2 * np.sqrt(var[i]), c, f"PC{i+1}: {var[i]/var.sum():.0%}")
    ax.set_aspect("equal"); ax.set_title("Главные компоненты — оси наибольшей дисперсии")
    ax = axes[1]
    Z = Xc @ Vt[0]
    ax.scatter(Z, np.zeros_like(Z), s=8, alpha=0.3, color=C[1])
    ax.hist(Z, bins=40, color=C[1], alpha=0.5)
    ax.set_title("Проекция на PC1: 2D → 1D, сохранено ~%d%% дисперсии" % round(100 * var[0] / var.sum()))
    ax.set_yticks([])
    save(fig, "pca")

pca()